## **DATA 620 - Project Three: Name Gender Classification**
## **Brandon Chanderban**

## Data Importation, Exploration, and Preparation

This project seeks to develop and evaluate a name-gender classifier using the NLTK Names Corpus, following the supervised classification techniques presented in Chapter 6 of Natural Language Processing with Python. The analysis will begin with a basic Naive Bayes classifier and progressively examine whether incorporating additional name characteristics improves classification accuracy.

The dataset will be divided into training, development-test, and test sets to facilitate model development and the independent evaluation of its final performance.

In [153]:
# Import libraries and download the NLTK Names Corpus.

import nltk
import random
from nltk.corpus import names
from collections import Counter

nltk.download("names")

# Examine the available corpus files.

print("The available corpus files are:")
print(names.fileids())

The available corpus files are:
['female.txt', 'male.txt']


[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


### Load and Inspect the names

The Names Corpus contains male and female name entries. As such, we shall combine both into a single labeled dataset.

In [154]:
# Load male and female names.

male_names = names.words("male.txt")
female_names = names.words("female.txt")

# Examine the number of names in each category.

print("The number of male names is", len(male_names))
print("The number of female names is", len(female_names))
print("The total number of names is", len(male_names) + len(female_names))

# Display a sample of the names.

print("\nThe first 10 male names are:")
print(male_names[:10])
print("\nThe first 10 female names are:")
print(female_names[:10])

The number of male names is 2943
The number of female names is 5001
The total number of names is 7944

The first 10 male names are:
['Aamir', 'Aaron', 'Abbey', 'Abbie', 'Abbot', 'Abbott', 'Abby', 'Abdel', 'Abdul', 'Abdulkarim']

The first 10 female names are:
['Abagael', 'Abagail', 'Abbe', 'Abbey', 'Abbi', 'Abbie', 'Abby', 'Abigael', 'Abigail', 'Abigale']


### Combine and Mix/Randomize the Dataset.

Our next step will involve creating a list of tuples containing each name and its associated classification label. Moreover, a random seed will be specified for reproducibility purposes.

In [155]:
# Create a combined dataset with gender labels.

labeled_names = (
    [(name, "male") for name in male_names]
    + [(name, "female") for name in female_names]
)

# Shuffle the names.

random.seed(6767)
random.shuffle(labeled_names)

# Inspect the first ten labeled entries.

print("Sample Labeled Names:")
for name, gender in labeled_names[:10]:
    print(name, "-", gender)

Sample Labeled Names:
Corinne - female
Apostolos - male
Myke - male
Yves - male
Gratiana - female
Eleni - female
Grace - male
Sam - female
Rosie - female
Nerta - female


### Inspect the Structure of the Labeled Dataset

Having combined and randomized the names, we will examine the structure and number of entries within the labeled dataset before proceeding with the training, development-test, and test splits.

In [156]:
# Examine the structure of the labeled dataset.

print("Dataset Type:", type(labeled_names))
print("Total Labeled Entries:", len(labeled_names))

print("\nFirst Five Labeled Entries:")
print(labeled_names[:5])

print("\nStructure of an Individual Entry:")
print(type(labeled_names[0]))
print(labeled_names[0])

# Examine the distribution of the gender labels.

gender_counts = Counter(gender for name, gender in labeled_names)

print("\nGender Distribution:")
print(gender_counts)

Dataset Type: <class 'list'>
Total Labeled Entries: 7944

First Five Labeled Entries:
[('Corinne', 'female'), ('Apostolos', 'male'), ('Myke', 'male'), ('Yves', 'male'), ('Gratiana', 'female')]

Structure of an Individual Entry:
<class 'tuple'>
('Corinne', 'female')

Gender Distribution:
Counter({'female': 5001, 'male': 2943})


### Split the Dataset

The subsequent step involves dividing the dataset into training, development-test, and test subsets.

As set out within the project instructions, 6,900 entries will be allocated to the training set, whilst 500 entries each will be reserved for the development-test and final test sets. Since the original corpus contains 7,944 labeled entries, 44 entries will be excluded to maintain the specified allocation.

The development-test set will be utilized to evaluate incremental improvements to the classifier, whereas the final test set will remain untouched until the selected model is ready for its final evaluation.

In [157]:
# Select 7,900 entries to match the project requirements.

selected_names = labeled_names[:7900]

# Split the dataset into the three required subsets.

test_names = selected_names[:500]
devtest_names = selected_names[500:1000]
train_names = selected_names[1000:]

# Verify the number of entries in each subset.

print("Number of records within the training set:", len(train_names))
print("Number of records within the development-test set:", len(devtest_names))
print("Number of records within the test set:", len(test_names))
print("Total Entries Used:", len(selected_names))
print("Unused Entries:", len(labeled_names) - len(selected_names))

# Confirm that the total matches the required allocation.

assert len(train_names) == 6900
assert len(devtest_names) == 500
assert len(test_names) == 500

Number of records within the training set: 6900
Number of records within the development-test set: 500
Number of records within the test set: 500
Total Entries Used: 7900
Unused Entries: 44


## Model 1: Establishing the Baseline Naive Bayes Classifier

With the training, development-test, and test subsets now prepared, the next step entails developing an initial Naive Bayes classifier using the approach demonstrated in Chapter 6 of Natural Language Processing with Python.

For this baseline model, the final letter of each name will be utilized as the sole classification feature. The classifier will be trained using the training subset and evaluated against the development-test subset to establish an initial measure of accuracy.

This baseline will subsequently serve as a reference for determining whether incorporating additional name characteristics improves classification performance.

### Defining the Feature Extraction Function

The same feature extraction function from within the textbook will be utilized.

In [158]:
# Define a feature extraction function using the final letter of each name.

def gender_features(word):
    return {"last_letter": word[-1].lower()}

# Examine the features extracted from sample names.

print("Features for Aaron:", gender_features("Aaron"))
print("Features for Abigail:", gender_features("Abigail"))
print("Features for Abdul:", gender_features("Abdul"))

Features for Aaron: {'last_letter': 'n'}
Features for Abigail: {'last_letter': 'l'}
Features for Abdul: {'last_letter': 'l'}


### Extract Features for the Training and Development-Test Sets

Subsequently, the function will be applied to each name whilst also preserving its corresponding gender label.

In [159]:
# Extract features for the training and development-test datasets.

train_set = [
    (gender_features(name), gender)
    for name, gender in train_names
]

devtest_set = [
    (gender_features(name), gender)
    for name, gender in devtest_names
]

# Examine the number of entries and a sample of the extracted features.

print("Training Feature Set:", len(train_set))
print("Development-Test Feature Set:", len(devtest_set))

print("\nSample Training Features:")
print(train_set[:5])

Training Feature Set: 6900
Development-Test Feature Set: 500

Sample Training Features:
[({'last_letter': 'y'}, 'female'), ({'last_letter': 'n'}, 'female'), ({'last_letter': 's'}, 'male'), ({'last_letter': 'n'}, 'female'), ({'last_letter': 'e'}, 'female')]


### Train the Naive Bayes Classifier

Next, NLTK's built-in classifier will be used as showcased within Chapter 6 of Natural Language Processing with Python.

In [160]:
# Train the baseline Naive Bayes classifier.

baseline_classifier = nltk.NaiveBayesClassifier.train(train_set)

# Evaluate the classifier on the development-test set.

baseline_accuracy = nltk.classify.accuracy(
    baseline_classifier, devtest_set
)

print("The Baseline Development-Test Accuracy:", round(baseline_accuracy, 4))
print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))

The Baseline Development-Test Accuracy: 0.76
The Baseline Accuracy (%): 76.0


### Examine the Sample Predictions

To better comprehend how the classifier operates, we can now examine its predictions for a small number of known name-gender classifications.

In [161]:
# Examine sample predictions from the baseline classifier.

sample_names = ["Aaron", "Abigail", "Abdul", "Gratiana", "Corinne"]

print("Sample Name Predictions:")

for name in sample_names:
    prediction = baseline_classifier.classify(gender_features(name))
    print(name, "-", prediction)

Sample Name Predictions:
Aaron - male
Abigail - female
Abdul - female
Gratiana - female
Corinne - female


### Examine the most Informative Features.

Now, we will inspect the features that most strongly distinguish the two classification labels.

In [162]:
# Display the most informative features identified by the classifier.

baseline_classifier.show_most_informative_features(10)

Most Informative Features
             last_letter = 'k'              male : female =     64.3 : 1.0
             last_letter = 'a'            female : male   =     34.5 : 1.0
             last_letter = 'f'              male : female =     15.8 : 1.0
             last_letter = 'p'              male : female =     11.1 : 1.0
             last_letter = 'd'              male : female =     10.5 : 1.0
             last_letter = 'm'              male : female =      9.3 : 1.0
             last_letter = 'v'              male : female =      9.1 : 1.0
             last_letter = 'o'              male : female =      8.9 : 1.0
             last_letter = 'z'              male : female =      7.0 : 1.0
             last_letter = 'r'              male : female =      6.2 : 1.0


The baseline Naive Bayes classifier achieved a development-test accuracy of 76.0%, suggesting that the final letter of a name provides a useful starting point for distinguishing between the male and female classifications within the NLTK Names Corpus.

However, certain limitations were evident in the sample predictions. For instance, whilst Aaron and Abigail were correctly classified, Abdul was incorrectly identified as female, demonstrating how names sharing the same final letter may belong to different gender categories.

Furthermore, the most informative features revealed strong associations between certain final letters and gender classifications, particularly "k" with male names and "a" with female names. In this regard, incorporating additional characteristics, such as prefixes, suffixes, and name length, may improve classification accuracy beyond the initial 76.0%.

## Incremental Improvements to the Naive Bayes Classifier

Following the initial development-test accuracy of 76.0%, the subsequent stage involves examining whether additional name characteristics can improve classification performance.

To accomplish this, several variations of the feature extraction function will be developed, incorporating longer suffixes, initial letters, and name length. Each variation will be trained using the same training subset and evaluated against the development-test subset, allowing for direct comparisons with the baseline classifier.

## Model 2: Utilizing the Final Two letters

For the first improvement, the feature extraction function will be modified to consider the final two letters of each name rather than only the final character. This will allow us to examine whether more detailed suffix patterns provide additional information for gender classification.

### Define the Feature Extraction Function

In [163]:
# Define a feature extraction function using the final two letters.

def gender_features_2(word):
    return {"last_two_letters": word[-2:].lower()}

# Examine the extracted features.

print("The features for Aaron:", gender_features_2("Aaron"))
print("The features for Abigail:", gender_features_2("Abigail"))
print("The features for Abdul:", gender_features_2("Abdul"))

The features for Aaron: {'last_two_letters': 'on'}
The features for Abigail: {'last_two_letters': 'il'}
The features for Abdul: {'last_two_letters': 'ul'}


### Train and Evaluate Model 2

In [164]:
# Extract features using the final two letters.

train_set_2 = [
    (gender_features_2(name), gender)
    for name, gender in train_names
]

devtest_set_2 = [
    (gender_features_2(name), gender)
    for name, gender in devtest_names
]

# Train the second Naive Bayes classifier.

classifier_2 = nltk.NaiveBayesClassifier.train(train_set_2)

# Evaluate Model 2 using the development-test set.

accuracy_2 = nltk.classify.accuracy(
    classifier_2, devtest_set_2
)

print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))
print("Model 2's Accuracy (%):", round(accuracy_2 * 100, 2))
print("Change in Accuracy (percentage points):",
      round((accuracy_2 - baseline_accuracy) * 100, 2))

The Baseline Accuracy (%): 76.0
Model 2's Accuracy (%): 81.6
Change in Accuracy (percentage points): 5.6


### Examine Sample Predictions

In [165]:
# Compare the baseline and Model 2 predictions.

print("Comparison of Sample Predictions:")

for name in sample_names:
    baseline_prediction = baseline_classifier.classify(
        gender_features(name)
    )

    model_2_prediction = classifier_2.classify(
        gender_features_2(name)
    )

    print(
        name,
        "| Baseline:", baseline_prediction,
        "| Model 2:", model_2_prediction
    )

Comparison of Sample Predictions:
Aaron | Baseline: male | Model 2: male
Abigail | Baseline: female | Model 2: female
Abdul | Baseline: female | Model 2: male
Gratiana | Baseline: female | Model 2: female
Corinne | Baseline: female | Model 2: female


The second Naive Bayes classifier achieved a development-test accuracy of 81.6%, representing an improvement of 5.6 percentage points compared with the baseline accuracy of 76.0%. This suggests that utilizing the final two letters of a name provides additional information that may improve gender classification.

Furthermore, the sample predictions revealed that Abdul, which was previously misclassified as female, was correctly identified as male under Model 2. In this regard, the findings demonstrate the potential benefits of incorporating more detailed suffix patterns, whilst providing justification for examining whether additional name characteristics could further improve classification performance.

## Model 3: Utilizing the Final Three Letters

For the subsequent modification, the feature extraction function will be adjusted to consider the final three letters of each name. This will allow us to investigate whether incorporating longer suffix patterns provides additional information for gender classification and results in improved accuracy when compared with the preceding models.

### Define the Feature Extraction Function

In [166]:
# Define a feature extraction function using the final three letters.

def gender_features_3(word):
    return {"last_three_letters": word[-3:].lower()}

# Examine the extracted features.

print("The features for Aaron:", gender_features_3("Aaron"))
print("The features for Abigail:", gender_features_3("Abigail"))
print("The features for Abdul:", gender_features_3("Abdul"))

The features for Aaron: {'last_three_letters': 'ron'}
The features for Abigail: {'last_three_letters': 'ail'}
The features for Abdul: {'last_three_letters': 'dul'}


### Train and Evaluate Model 3

In [167]:
# Extract features using the final three letters.

train_set_3 = [
    (gender_features_3(name), gender)
    for name, gender in train_names
]

devtest_set_3 = [
    (gender_features_3(name), gender)
    for name, gender in devtest_names
]

# Train the third Naive Bayes classifier.

classifier_3 = nltk.NaiveBayesClassifier.train(train_set_3)

# Evaluate Model 3 using the development-test set.

accuracy_3 = nltk.classify.accuracy(
    classifier_3, devtest_set_3
)

print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))
print("Model 2's Accuracy (%):", round(accuracy_2 * 100, 2))
print("Model 3's Accuracy (%):", round(accuracy_3 * 100, 2))

print(
    "Change from Model 2 (percentage points):",
    round((accuracy_3 - accuracy_2) * 100, 2)
)

print(
    "Change from Baseline (percentage points):",
    round((accuracy_3 - baseline_accuracy) * 100, 2)
)

The Baseline Accuracy (%): 76.0
Model 2's Accuracy (%): 81.6
Model 3's Accuracy (%): 76.6
Change from Model 2 (percentage points): -5.0
Change from Baseline (percentage points): 0.6


### Compare the Sample Predictions

In [168]:
# Compare predictions across the three models.

print("Comparison of Sample Predictions:")

for name in sample_names:
    baseline_prediction = baseline_classifier.classify(
        gender_features(name)
    )

    model_2_prediction = classifier_2.classify(
        gender_features_2(name)
    )

    model_3_prediction = classifier_3.classify(
        gender_features_3(name)
    )

    print(
        name,
        "| Baseline:", baseline_prediction,
        "| Model 2:", model_2_prediction,
        "| Model 3:", model_3_prediction
    )

Comparison of Sample Predictions:
Aaron | Baseline: male | Model 2: male | Model 3: male
Abigail | Baseline: female | Model 2: female | Model 3: male
Abdul | Baseline: female | Model 2: male | Model 3: male
Gratiana | Baseline: female | Model 2: female | Model 3: female
Corinne | Baseline: female | Model 2: female | Model 3: female


The third Naive Bayes classifier achieved a development-test accuracy of 76.6%, representing a decrease of 5.0 percentage points compared with Model 2's accuracy of 81.6%. However, this remained slightly above the baseline accuracy of 76.0%.

Furthermore, the sample predictions revealed that whilst Abdul remained correctly classified as male, Abigail was incorrectly identified as male under Model 3, despite being correctly classified by the preceding models. In this regard, the findings suggest that utilizing longer suffixes does not necessarily improve classification performance, potentially because more specific letter combinations occur less frequently within the training data.

Consequently, Model 2 remains the strongest-performing classifier thus far, providing justification for exploring combinations of different name characteristics rather than continuing to increase suffix length.

## Model 4: Combining the First Letter and Final Two Letters

Since Model 2 achieved a higher development-test accuracy than Model 3, the subsequent modification will involve combining the first letter of each name with its final two letters.

This approach will allow us to investigate whether incorporating information from both the beginning and ending of a name provides additional discriminatory value, potentially improving classification accuracy beyond the 81.6% achieved by Model 2.

### Define the Feature Extraction Function

In [169]:
# Define a feature extraction function using the first letter
# and final two letters of each name.

def gender_features_4(word):
    return {
        "first_letter": word[0].lower(),
        "last_two_letters": word[-2:].lower()
    }

# Examine the extracted features.

print("The features for Aaron:", gender_features_4("Aaron"))
print("The features for Abigail:", gender_features_4("Abigail"))
print("The features for Abdul:", gender_features_4("Abdul"))

The features for Aaron: {'first_letter': 'a', 'last_two_letters': 'on'}
The features for Abigail: {'first_letter': 'a', 'last_two_letters': 'il'}
The features for Abdul: {'first_letter': 'a', 'last_two_letters': 'ul'}


### Train and Evaluate Model 4

The fourth Naive Bayes classifier will be trained using the existing training subset and evaluated against the same development-test subset. Its accuracy will then be compared with the preceding models, particularly Model 2, to determine whether incorporating the first-letter feature improves classification performance.

In [170]:
# Extract features using the first letter and final two letters.

train_set_4 = [
    (gender_features_4(name), gender)
    for name, gender in train_names
]

devtest_set_4 = [
    (gender_features_4(name), gender)
    for name, gender in devtest_names
]

# Train the fourth Naive Bayes classifier.

classifier_4 = nltk.NaiveBayesClassifier.train(train_set_4)

# Evaluate Model 4 using the development-test set.

accuracy_4 = nltk.classify.accuracy(
    classifier_4, devtest_set_4
)

print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))
print("Model 2's Accuracy (%):", round(accuracy_2 * 100, 2))
print("Model 3's Accuracy (%):", round(accuracy_3 * 100, 2))
print("Model 4's Accuracy (%):", round(accuracy_4 * 100, 2))

print(
    "Change from Model 2 (percentage points):",
    round((accuracy_4 - accuracy_2) * 100, 2)
)

print(
    "Change from Baseline (percentage points):",
    round((accuracy_4 - baseline_accuracy) * 100, 2)
)

The Baseline Accuracy (%): 76.0
Model 2's Accuracy (%): 81.6
Model 3's Accuracy (%): 76.6
Model 4's Accuracy (%): 81.6
Change from Model 2 (percentage points): 0.0
Change from Baseline (percentage points): 5.6


### Compare Predictions Across All Four Models

In [171]:
# Compare predictions across the four models.

print("Comparison of Sample Predictions:")

for name in sample_names:
    baseline_prediction = baseline_classifier.classify(
        gender_features(name)
    )

    model_2_prediction = classifier_2.classify(
        gender_features_2(name)
    )

    model_3_prediction = classifier_3.classify(
        gender_features_3(name)
    )

    model_4_prediction = classifier_4.classify(
        gender_features_4(name)
    )

    print(
        name,
        "| Baseline:", baseline_prediction,
        "| Model 2:", model_2_prediction,
        "| Model 3:", model_3_prediction,
        "| Model 4:", model_4_prediction
    )

Comparison of Sample Predictions:
Aaron | Baseline: male | Model 2: male | Model 3: male | Model 4: male
Abigail | Baseline: female | Model 2: female | Model 3: male | Model 4: female
Abdul | Baseline: female | Model 2: male | Model 3: male | Model 4: male
Gratiana | Baseline: female | Model 2: female | Model 3: female | Model 4: female
Corinne | Baseline: female | Model 2: female | Model 3: female | Model 4: female


The fourth Naive Bayes classifier achieved a development-test accuracy of 81.6%, matching the performance of Model 2 and representing an improvement of 5.6 percentage points compared with the baseline.

Furthermore, the sample predictions remained consistent with those generated by Model 2, with all five selected names correctly classified according to their respective labels within the Names Corpus. In this regard, incorporating the first letter alongside the final two letters did not produce any additional improvement in classification accuracy, suggesting that the first-letter feature may contribute limited supplementary information.

Nevertheless, these findings provide a useful basis for examining whether introducing another characteristic, such as name length, could further improve the classifier's performance.

## Model 5: Combining the First Letter, Final Two Letters, and Name Length

Following the findings from Model 4, which demonstrated that incorporating the first letter of a name did not improve classification accuracy beyond Model 2, the subsequent modification will involve introducing name length as an additional feature.

This approach will allow us to investigate whether combining the initial letter, final two letters, and total number of characters provides supplementary information for gender classification, potentially improving accuracy beyond the 81.6% achieved by Models 2 and 4.

### Define the Feature Extraction Function

In [172]:
# Define a feature extraction function using the first letter,
# final two letters, and name length.

def gender_features_5(word):
    return {
        "first_letter": word[0].lower(),
        "last_two_letters": word[-2:].lower(),
        "name_length": len(word)
    }

# Examine the extracted features.

print("The features for Aaron:", gender_features_5("Aaron"))
print("The features for Abigail:", gender_features_5("Abigail"))
print("The features for Abdul:", gender_features_5("Abdul"))

The features for Aaron: {'first_letter': 'a', 'last_two_letters': 'on', 'name_length': 5}
The features for Abigail: {'first_letter': 'a', 'last_two_letters': 'il', 'name_length': 7}
The features for Abdul: {'first_letter': 'a', 'last_two_letters': 'ul', 'name_length': 5}


### Train and Evaluate Model 5

In [173]:
# Extract features using the first letter, final two letters,
# and name length.

train_set_5 = [
    (gender_features_5(name), gender)
    for name, gender in train_names
]

devtest_set_5 = [
    (gender_features_5(name), gender)
    for name, gender in devtest_names
]

# Train the fifth Naive Bayes classifier.

classifier_5 = nltk.NaiveBayesClassifier.train(train_set_5)

# Evaluate Model 5 using the development-test set.

accuracy_5 = nltk.classify.accuracy(
    classifier_5, devtest_set_5
)

print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))
print("Model 2's Accuracy (%):", round(accuracy_2 * 100, 2))
print("Model 3's Accuracy (%):", round(accuracy_3 * 100, 2))
print("Model 4's Accuracy (%):", round(accuracy_4 * 100, 2))
print("Model 5's Accuracy (%):", round(accuracy_5 * 100, 2))

print(
    "Change from Model 4 (percentage points):",
    round((accuracy_5 - accuracy_4) * 100, 2)
)

print(
    "Change from Baseline (percentage points):",
    round((accuracy_5 - baseline_accuracy) * 100, 2)
)

The Baseline Accuracy (%): 76.0
Model 2's Accuracy (%): 81.6
Model 3's Accuracy (%): 76.6
Model 4's Accuracy (%): 81.6
Model 5's Accuracy (%): 81.6
Change from Model 4 (percentage points): 0.0
Change from Baseline (percentage points): 5.6


### Compare Sample Predictions

To examine whether the introduction of name length influences individual classifications, the predictions generated by Model 5 will be compared with those of the preceding models using the same sample names.

In [174]:
# Compare sample predictions across all five models.

import pandas as pd

prediction_results = []

for name in sample_names:
    prediction_results.append({
        "Name": name,
        "Baseline": baseline_classifier.classify(gender_features(name)),
        "Model 2": classifier_2.classify(gender_features_2(name)),
        "Model 3": classifier_3.classify(gender_features_3(name)),
        "Model 4": classifier_4.classify(gender_features_4(name)),
        "Model 5": classifier_5.classify(gender_features_5(name))
    })

prediction_df = pd.DataFrame(prediction_results)

display(prediction_df)

,Name,Baseline,Model 2,Model 3,Model 4,Model 5
0,Aaron,male,male,male,male,male
1,Abigail,female,female,male,female,female
2,Abdul,female,male,male,male,male
3,Gratiana,female,female,female,female,female
4,Corinne,female,female,female,female,female


The fifth Naive Bayes classifier achieved a development-test accuracy of 81.6%, matching the results of Models 2 and 4, and representing an improvement of 5.6 percentage points compared with the baseline.

Despite incorporating name length as an additional feature, no further improvement in accuracy was observed. In this regard, the findings suggest that name length may provide limited supplementary value when combined with the first letter and final two letters of a name, at least within the current development-test dataset.

Consequently, the results reinforce the importance of evaluating feature modifications rather than assuming that incorporating additional characteristics will necessarily improve classification performance.

## Model 6: Combining the Final Two Letters and Vowel Count

For the sixth model, the feature extraction function will be modified to incorporate the number of vowels present within each name alongside its final two letters.

This approach seeks to investigate whether vowel frequency provides additional information for gender classification beyond the suffix patterns examined previously. By retaining the final two letters utilized in Model 2, we can determine whether introducing vowel count improves classification accuracy beyond the current highest development-test accuracy of 81.6%.

### Define the Feature Extraction Function

In this, our goal is to count the standard English vowels (a, e, i, o, u) within each name.

In [175]:
# Define a feature extraction function using the final two letters
# and the number of vowels within each name.

def gender_features_6(word):
    vowels = "aeiou"
    vowel_count = sum(1 for letter in word.lower() if letter in vowels)

    return {
        "last_two_letters": word[-2:].lower(),
        "vowel_count": vowel_count
    }

# Examine the extracted features.

print("The features for Aaron:", gender_features_6("Aaron"))
print("The features for Abigail:", gender_features_6("Abigail"))
print("The features for Abdul:", gender_features_6("Abdul"))

The features for Aaron: {'last_two_letters': 'on', 'vowel_count': 3}
The features for Abigail: {'last_two_letters': 'il', 'vowel_count': 4}
The features for Abdul: {'last_two_letters': 'ul', 'vowel_count': 2}


### Train and Evaluate Model 6

The sixth Naive Bayes classifier will be trained using the same training subset and evaluated against the existing development-test subset. Its accuracy will then be compared with Model 2 and the baseline to determine whether incorporating vowel count contributes to improved classification performance.

In [176]:
# Extract features using the final two letters and vowel count.

train_set_6 = [
    (gender_features_6(name), gender)
    for name, gender in train_names
]

devtest_set_6 = [
    (gender_features_6(name), gender)
    for name, gender in devtest_names
]

# Train the sixth Naive Bayes classifier.

classifier_6 = nltk.NaiveBayesClassifier.train(train_set_6)

# Evaluate Model 6 using the development-test set.

accuracy_6 = nltk.classify.accuracy(
    classifier_6, devtest_set_6
)

print("The Baseline Accuracy (%):", round(baseline_accuracy * 100, 2))
print("Model 2's Accuracy (%):", round(accuracy_2 * 100, 2))
print("Model 3's Accuracy (%):", round(accuracy_3 * 100, 2))
print("Model 4's Accuracy (%):", round(accuracy_4 * 100, 2))
print("Model 5's Accuracy (%):", round(accuracy_5 * 100, 2))
print("Model 6's Accuracy (%):", round(accuracy_6 * 100, 2))

print(
    "\nChange from Model 2 (percentage points):",
    round((accuracy_6 - accuracy_2) * 100, 2)
)

print(
    "Change from Baseline (percentage points):",
    round((accuracy_6 - baseline_accuracy) * 100, 2)
)

The Baseline Accuracy (%): 76.0
Model 2's Accuracy (%): 81.6
Model 3's Accuracy (%): 76.6
Model 4's Accuracy (%): 81.6
Model 5's Accuracy (%): 81.6
Model 6's Accuracy (%): 79.2

Change from Model 2 (percentage points): -2.4
Change from Baseline (percentage points): 3.2


### Compare Sample Predictions

To investigate whether the introduction of vowel count influences individual classifications, the predictions generated by Model 6 will be compared with those of the preceding models using the same five sample names.

In [177]:
# Compare sample predictions across all six models.

import pandas as pd

prediction_results = []

for name in sample_names:
    prediction_results.append({
        "Name": name,
        "Baseline": baseline_classifier.classify(gender_features(name)),
        "Model 2": classifier_2.classify(gender_features_2(name)),
        "Model 3": classifier_3.classify(gender_features_3(name)),
        "Model 4": classifier_4.classify(gender_features_4(name)),
        "Model 5": classifier_5.classify(gender_features_5(name)),
        "Model 6": classifier_6.classify(gender_features_6(name))
    })

prediction_df = pd.DataFrame(prediction_results)

display(prediction_df)

,Name,Baseline,Model 2,Model 3,Model 4,Model 5,Model 6
0,Aaron,male,male,male,male,male,male
1,Abigail,female,female,male,female,female,female
2,Abdul,female,male,male,male,male,male
3,Gratiana,female,female,female,female,female,female
4,Corinne,female,female,female,female,female,female


The sixth Naive Bayes classifier achieved a development-test accuracy of 79.2%, representing a decrease of 2.4 percentage points compared with Model 2's accuracy of 81.6%, although remaining 3.2 percentage points above the baseline.

In this regard, incorporating vowel count alongside the final two letters did not produce the anticipated improvement in classification performance. This suggests that whilst vowel frequency may provide supplementary information regarding name characteristics, its inclusion was not beneficial to the classifier's accuracy within the current development-test dataset.

Consequently, these findings further demonstrate that introducing additional features does not necessarily translate into improved classification performance.

## Classification-Error Analysis

Following the development and evaluation of six Naive Bayes classifiers, the subsequent stage involves examining classification errors associated with the highest-performing models.

Since Models 2, 4, and 5 each achieved a development-test accuracy of 81.6%, Model 2 will be utilized for this analysis due to its simpler feature extraction function.

By comparing predicted classifications with the actual gender labels recorded within the development-test subset, we can identify names that were incorrectly classified and investigate whether certain suffix patterns contributed to these errors.

### Identify Incorrectly Classified Names

In [178]:
# Identify incorrectly classified names in the development-test set.

classification_errors = []

for name, actual_gender in devtest_names:
    predicted_gender = classifier_2.classify(
        gender_features_2(name)
    )

    if predicted_gender != actual_gender:
        classification_errors.append({
            "Name": name,
            "Actual Gender": actual_gender,
            "Predicted Gender": predicted_gender,
            "Final Two Letters": name[-2:].lower()
        })

# Convert the classification errors into a DataFrame.

import pandas as pd

errors_df = pd.DataFrame(classification_errors)

print("Total Development-Test Entries:", len(devtest_names))
print("Incorrect Classifications:", len(errors_df))
print("Correct Classifications:", len(devtest_names) - len(errors_df))

print("\nSample Classification Errors:")
display(errors_df.head(15))

Total Development-Test Entries: 500
Incorrect Classifications: 92
Correct Classifications: 408

Sample Classification Errors:


,Name,Actual Gender,Predicted Gender,Final Two Letters
0,Ken,male,female,en
1,Rosalynd,female,male,nd
2,Lark,female,male,rk
3,Willey,male,female,ey
4,Sherill,female,male,ll
5,Kirby,female,male,by
6,Billie,male,female,ie
7,Sturgis,male,female,is
8,Carmon,female,male,on
9,Frank,female,male,nk


### Examine Patterns in the Errors

Subsequently, we can now identify which of the two-letter suffixes appear most frequently amongst the misclassified names.

In [179]:
# Examine the most common suffixes among misclassified names.

print("Most Common Suffixes Among Classification Errors:")

print(
    errors_df["Final Two Letters"]
    .value_counts()
    .head(10)
)

# Examine the distribution of actual labels among errors.

print("\nClassification Errors by Actual Gender:")

print(errors_df["Actual Gender"].value_counts())

Most Common Suffixes Among Classification Errors:
Final Two Letters
ie    9
en    6
ry    5
is    4
an    4
le    3
il    3
tt    3
by    2
re    2
Name: count, dtype: int64

Classification Errors by Actual Gender:
Actual Gender
male      63
female    29
Name: count, dtype: int64


The classification-error analysis revealed that Model 2 incorrectly classified 92 of the 500 development-test entries, corresponding to an error rate of 18.4%. Of these, 63 involved names labeled as male, whilst the remaining 29 involved names labeled as female.

Furthermore, examining the suffixes associated with these errors revealed that "ie" appeared most frequently, accounting for nine misclassifications, followed by "en" with six and "ry" with five. In this regard, the findings suggest that certain two-letter endings may be less effective at distinguishing between gender classifications, particularly when similar suffixes occur across both categories.

These results highlight the limitations of relying exclusively on suffix patterns and provide additional context for interpreting the classifier's development-test performance.

## Final Model Selection and Evaluation

Following the development and evaluation of six Naive Bayes classifiers, Model 2 was selected for final testing based on its development-test accuracy of 81.6%, which matched the highest performance achieved by Models 4 and 5.

Although these three models demonstrated identical accuracy, Model 2 utilizes only the final two letters of each name, making it the simpler alternative.

The selected classifier will now be evaluated against the previously reserved test subset containing 500 entries. Its performance will then be compared with the development-test results to determine whether the classifier demonstrates similar accuracy when applied to observations that were not utilized during training or model selection.

### Prepare the Final Test Set

Model 2's existing feature extraction function, gender_features_2(), will now be applied to the reserved test entries.

In [180]:
# Extract features from the final test set using Model 2.

test_set_2 = [
    (gender_features_2(name), gender)
    for name, gender in test_names
]

# Confirm the number of entries in the test set.

print("Final Test Set Size:", len(test_set_2))

print("\nSample Test Features:")
print(test_set_2[:5])

Final Test Set Size: 500

Sample Test Features:
[({'last_two_letters': 'ne'}, 'female'), ({'last_two_letters': 'os'}, 'male'), ({'last_two_letters': 'ke'}, 'male'), ({'last_two_letters': 'es'}, 'male'), ({'last_two_letters': 'na'}, 'female')]


### Evaluate the Selected Classifier

The accuracy of Model 2 on the final test set will then be calculated and compared directly with its development-test accuracy.

In [181]:
# Evaluate the selected classifier on the final test set.

final_test_accuracy = nltk.classify.accuracy(
    classifier_2, test_set_2
)

# Compare the development-test and final test accuracies.

print("Selected Classifier: Model 2 (Final Two Letters)")

print(
    "Development-Test Accuracy (%):",
    round(accuracy_2 * 100, 2)
)

print(
    "Final Test Accuracy (%):",
    round(final_test_accuracy * 100, 2)
)

print(
    "Difference (percentage points):",
    round((final_test_accuracy - accuracy_2) * 100, 2)
)

Selected Classifier: Model 2 (Final Two Letters)
Development-Test Accuracy (%): 81.6
Final Test Accuracy (%): 79.8
Difference (percentage points): -1.8


### Summarize the Classification Results

In [182]:
# Summarize the selected classifier's final performance.

correct_predictions = sum(
    classifier_2.classify(features) == actual_gender
    for features, actual_gender in test_set_2
)

incorrect_predictions = len(test_set_2) - correct_predictions

print("The final test results are as follows:")
print("Total Test Entries:", len(test_set_2))
print("Number of Correct Classifications:", correct_predictions)
print("Number of Incorrect Classifications:", incorrect_predictions)
print("The Final Test Accuracy (%):", round(final_test_accuracy * 100, 2))

The final test results are as follows:
Total Test Entries: 500
Number of Correct Classifications: 399
Number of Incorrect Classifications: 101
The Final Test Accuracy (%): 79.8


The selected Naive Bayes classifier achieved a final test accuracy of 79.8%, correctly classifying 399 of the 500 reserved test entries, whilst incorrectly classifying the remaining 101. This represents a decrease of 1.8 percentage points compared with the development-test accuracy of 81.6%.

The relatively small difference between these results was reasonably expected, given that the two subsets contain different observations and that some variation in classification performance is inevitable. Furthermore, since the development-test set was utilized throughout the model refinement process, its accuracy may have been slightly more favorable than that obtained from the independent test set.

In this regard, the findings suggest that Model 2 demonstrated reasonably consistent classification performance when applied to previously unseen entries, supporting its selection as the preferred model amongst the six variations examined.

## Findings and Conclusions

The development and evaluation of six Naive Bayes classifiers demonstrated how different name characteristics can influence gender classification accuracy. The development-test results are summarized below.

In [183]:
# Summarize the development-test accuracy of all six models.

model_comparison = pd.DataFrame({
    "Model": [
        "Model 1 (Baseline)",
        "Model 2",
        "Model 3",
        "Model 4",
        "Model 5",
        "Model 6"
    ],
    "Features Utilized": [
        "Final letter",
        "Final two letters",
        "Final three letters",
        "First letter and final two letters",
        "First letter, final two letters, and name length",
        "Final two letters and vowel count"
    ],
    "Development-Test Accuracy (%)": [
        baseline_accuracy * 100,
        accuracy_2 * 100,
        accuracy_3 * 100,
        accuracy_4 * 100,
        accuracy_5 * 100,
        accuracy_6 * 100
    ]
})

# Round accuracy values to two decimal places.

model_comparison["Development-Test Accuracy (%)"] = (
    model_comparison["Development-Test Accuracy (%)"].round(2)
)

# Display the comparison table.

display(model_comparison)

,Model,Features Utilized,Development-Test Accuracy (%)
0,Model 1 (Baseline),Final letter,76.0
1,Model 2,Final two letters,81.6
2,Model 3,Final three letters,76.6
3,Model 4,First letter and final two letters,81.6
4,Model 5,"First letter, final two letters, and name length",81.6
5,Model 6,Final two letters and vowel count,79.2


The findings indicate that incorporating additional features does not necessarily improve classification performance. Although Models 2, 4, and 5 achieved the highest development-test accuracy of 81.6%, Model 2 was selected due to its comparatively simpler feature extraction approach. Its final test accuracy of 79.8%, representing a decrease of 1.8 percentage points, was reasonably consistent with the development-test results and within expectations.

Furthermore, the presence of certain names within both gender categories of the corpus, such as Abbey, Abbie, and Abby, which appeared amongst the first ten entries of both the male and female lists during our preliminary exploration, highlights a limitation of relying exclusively on name spellings for gender classification.

In conclusion, the project demonstrated the importance of incremental feature development and independent model evaluation. Whilst suffix patterns proved useful, the findings also emphasize that the resulting classifier should be understood as a means of predicting labels within the NLTK Names Corpus rather than determining an individual's gender identity.